In [30]:
%%sql

select count(*) FROM bronze_nyc_trips_raw;
select count(*) FROM bronze_weather_raw;

StatementMeta(, 71c697d9-686a-44e4-bd02-c83e5f6d903f, 32, Finished, Available, Finished)

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

In [3]:
%%sql
DESCRIBE TABLE bronze_nyc_trips_raw;


StatementMeta(, 796c1528-b4bb-4ac5-8452-fc2ce0205ebc, 7, Finished, Available, Finished)

<Spark SQL result set with 20 rows and 3 fields>

In [4]:
%%sql
CREATE OR REPLACE TABLE silver_nyc_trips_clean AS
SELECT 
  VendorID,
  tpep_pickup_datetime,
  tpep_dropoff_datetime,
  passenger_count,
  trip_distance,
  RatecodeID,
  store_and_fwd_flag,
  PULocationID,
  DOLocationID,
  payment_type,
  fare_amount,
  extra,
  mta_tax,
  tip_amount,
  tolls_amount,
  improvement_surcharge,
  total_amount,
  congestion_surcharge,
  Airport_fee,
  cbd_congestion_fee,
  DATE_TRUNC('day', tpep_pickup_datetime) as pickup_date,
  DATE_TRUNC('month', tpep_pickup_datetime) as pickup_month
FROM bronze_nyc_trips_raw
WHERE 1=1
  AND fare_amount > 0
  AND total_amount > 0
  AND passenger_count BETWEEN 1 AND 6
  AND trip_distance BETWEEN 0.01 AND 100
  AND tpep_pickup_datetime IS NOT NULL
  AND tpep_dropoff_datetime IS NOT NULL
  AND tpep_dropoff_datetime > tpep_pickup_datetime
  AND improvement_surcharge BETWEEN 0 AND 5
  AND PULocationID BETWEEN 1 AND 265
  AND DOLocationID BETWEEN 1 AND 265


StatementMeta(, 796c1528-b4bb-4ac5-8452-fc2ce0205ebc, 8, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

In [5]:
%%sql
select count(*) from silver_nyc_trips_clean

StatementMeta(, 796c1528-b4bb-4ac5-8452-fc2ce0205ebc, 9, Finished, Available, Finished)

<Spark SQL result set with 1 rows and 1 fields>

In [6]:
%%sql

SELECT 
  COUNT(*) as total_rows,
  COUNT(DISTINCT PULocationID) as unique_pu_locations,
  COUNT(DISTINCT DOLocationID) as unique_do_locations,
  AVG(trip_distance) as avg_distance,
  MIN(fare_amount) as min_fare,
  MAX(fare_amount) as max_fare
FROM silver_nyc_trips_clean;

StatementMeta(, 796c1528-b4bb-4ac5-8452-fc2ce0205ebc, 10, Finished, Available, Finished)

<Spark SQL result set with 1 rows and 6 fields>

In [20]:
%%sql
DESCRIBE TABLE bronze_weather_raw;

StatementMeta(, 796c1528-b4bb-4ac5-8452-fc2ce0205ebc, 24, Finished, Available, Finished)

<Spark SQL result set with 7 rows and 3 fields>

In [35]:
%%sql
CREATE OR REPLACE TABLE silver_weather_daily AS
SELECT 
  TO_DATE(Date, 'dd-MM-yyyy') as weather_date,
  `TAVG (Degrees Fahrenheit)` as avg_temp_f,
  `TMAX (Degrees Fahrenheit)` as max_temp_f,
  `TMIN (Degrees Fahrenheit)` as min_temp_f,
  `PRCP (Inches)` as precip_in,
  `SNOW (Inches)` as snowfall_in,
  `SNWD (Inches)` as snow_depth_in
FROM bronze_weather_raw;


StatementMeta(, 796c1528-b4bb-4ac5-8452-fc2ce0205ebc, 39, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

In [36]:
%%sql
SELECT COUNT(*) as silver_weather_rows, 
       MIN(weather_date), MAX(weather_date),
       AVG(avg_temp_f) as avg_temp
FROM silver_weather_daily;


StatementMeta(, 796c1528-b4bb-4ac5-8452-fc2ce0205ebc, 40, Finished, Available, Finished)

<Spark SQL result set with 1 rows and 4 fields>

In [1]:
%%sql
-- NYC Trips range
SELECT 'NYC_Trips' as table_name, 
       MIN(tpep_pickup_datetime) as min_date,
       MAX(tpep_pickup_datetime) as max_date,
       COUNT(*) as row_count
FROM silver_nyc_trips_clean

UNION ALL

-- Weather range  
SELECT 'Weather' as table_name,
       MIN(weather_date) as min_date,
       MAX(weather_date) as max_date,
       COUNT(*) as row_count
FROM silver_weather_daily



StatementMeta(, 41d935d8-82f9-44ec-94fe-7f0b007a15f5, 2, Finished, Available, Finished)

<Spark SQL result set with 2 rows and 4 fields>

# Gold Layer Tables

In [3]:
%%sql
CREATE OR REPLACE TABLE gold_zone_day_metrics AS
WITH weather_cast AS (
    SELECT
        CAST(weather_date AS date)                  AS weather_date,
        CAST(avg_temp_f     AS double)              AS avg_temp_f_num,
        CAST(max_temp_f     AS double)              AS max_temp_f_num,
        CAST(min_temp_f     AS double)              AS min_temp_f_num,
        CAST(precip_in      AS double)              AS precip_in_num,
        CAST(snowfall_in    AS double)              AS snowfall_in_num,
        CAST(snow_depth_in  AS double)              AS snow_depth_in_num
    FROM silver_weather_daily
),
trips_enriched AS (
    SELECT
        PULocationID                                        AS zone_id,
        CAST(tpep_pickup_datetime AS date)                  AS service_date,
        dayofweek(tpep_pickup_datetime)                     AS day_of_week,  
        CASE 
            WHEN dayofweek(tpep_pickup_datetime) IN (1, 7) 
                THEN TRUE 
            ELSE FALSE 
        END                                                 AS is_weekend,
        passenger_count,
        trip_distance,
        tpep_pickup_datetime,
        tpep_dropoff_datetime,
        fare_amount,
        extra,
        mta_tax,
        tip_amount,
        tolls_amount,
        improvement_surcharge,
        total_amount,
        congestion_surcharge,
        Airport_fee,
        cbd_congestion_fee,
        payment_type
    FROM silver_nyc_trips_clean
)
SELECT
    t.zone_id,
    t.service_date,
    t.day_of_week,
    t.is_weekend,

    COUNT(*)                                     AS trips,
    SUM(t.passenger_count)                       AS passenger_count_total,
    AVG(t.passenger_count)                       AS avg_passenger_count,

    SUM(t.total_amount)                          AS total_revenue,
    SUM(t.fare_amount)                           AS total_fare,
    CASE WHEN COUNT(*) > 0 
         THEN SUM(t.fare_amount) / COUNT(*) 
    END                                          AS avg_fare,
    CASE WHEN COUNT(*) > 0 
         THEN SUM(t.total_amount) / COUNT(*) 
    END                                          AS avg_total_amount,
    SUM(t.tip_amount)                            AS tip_amount_total,
    CASE WHEN SUM(t.total_amount) <> 0 
         THEN SUM(t.tip_amount) / SUM(t.total_amount)
    END                                          AS tip_share_pct,

    AVG(t.trip_distance)                         AS avg_trip_distance,
    AVG(
        (unix_timestamp(t.tpep_dropoff_datetime) 
       - unix_timestamp(t.tpep_pickup_datetime)) / 60.0
    )                                            AS avg_trip_duration_min,

    SUM(CASE WHEN t.payment_type = 1 THEN 1 ELSE 0 END) AS card_trips,
    SUM(CASE WHEN t.payment_type = 2 THEN 1 ELSE 0 END) AS cash_trips,
    CASE WHEN COUNT(*) > 0 
         THEN SUM(CASE WHEN t.payment_type = 1 THEN 1 ELSE 0 END) / COUNT(*) 
    END                                          AS card_share_pct,

    w.avg_temp_f_num,
    w.max_temp_f_num,
    w.min_temp_f_num,
    w.precip_in_num,
    w.snowfall_in_num,
    w.snow_depth_in_num,
    CASE WHEN w.precip_in_num   > 0 THEN TRUE ELSE FALSE END AS is_rain,
    CASE WHEN w.snowfall_in_num > 0 THEN TRUE ELSE FALSE END AS is_snow

FROM trips_enriched t
LEFT JOIN weather_cast w
    ON t.service_date = w.weather_date
GROUP BY
    t.zone_id,
    t.service_date,
    t.day_of_week,
    t.is_weekend,
    w.avg_temp_f_num,
    w.max_temp_f_num,
    w.min_temp_f_num,
    w.precip_in_num,
    w.snowfall_in_num,
    w.snow_depth_in_num;


StatementMeta(, 98b9885f-982a-49b4-92ad-a965e1983a68, 4, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

In [4]:
%%sql
SELECT 
  MIN(service_date) AS min_date,
  MAX(service_date) AS max_date,
  COUNT(*)          AS rows
FROM gold_zone_day_metrics;


StatementMeta(, 98b9885f-982a-49b4-92ad-a965e1983a68, 5, Finished, Available, Finished)

<Spark SQL result set with 1 rows and 3 fields>

In [5]:
%%sql
SELECT zone_id,
       SUM(trips)           AS total_trips,
       SUM(total_revenue)   AS total_revenue
FROM gold_zone_day_metrics
GROUP BY zone_id
ORDER BY total_revenue DESC
LIMIT 10;


StatementMeta(, 98b9885f-982a-49b4-92ad-a965e1983a68, 6, Finished, Available, Finished)

<Spark SQL result set with 10 rows and 3 fields>

In [2]:
%%sql
CREATE OR REPLACE TABLE gold_zone_hour_metrics AS
SELECT
    date(t.tpep_pickup_datetime)                     AS pickup_date,
    hour(t.tpep_pickup_datetime)                     AS pickup_hour,
    t.PULocationID                                   AS zone_id,

    COUNT(*)                                         AS trips_count,
    SUM(t.total_amount)                              AS total_revenue,
    CASE WHEN COUNT(*) > 0
         THEN SUM(t.total_amount) / COUNT(*)
    END                                              AS avg_total_amount,

    AVG(t.trip_distance)                             AS avg_trip_distance,
    AVG(
        (unix_timestamp(t.tpep_dropoff_datetime)
       - unix_timestamp(t.tpep_pickup_datetime)) / 60.0
    )                                                AS avg_trip_duration_min,

    SUM(t.tip_amount)                                AS tip_amount_total,
    CASE WHEN SUM(t.total_amount) <> 0
         THEN SUM(t.tip_amount) / SUM(t.total_amount)
    END                                              AS tip_share_pct,

    CASE WHEN dayofweek(t.tpep_pickup_datetime) IN (1, 7)
         THEN TRUE ELSE FALSE
    END                                              AS is_weekend
FROM silver_nyc_trips_clean t
GROUP BY
    date(t.tpep_pickup_datetime),
    hour(t.tpep_pickup_datetime),
    t.PULocationID;


StatementMeta(, 22a0bc23-e66a-4704-8eb7-d4f9492dfb7d, 3, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

# Sanity Checks
**1. Row‑count consistency vs. silver**


In [4]:
%%sql
SELECT
  (SELECT COUNT(*) FROM silver_nyc_trips_clean)                               AS silver_trips,
  (SELECT COALESCE(SUM(trips_count), 0) FROM gold_zone_hour_metrics)      AS gold_hour_trips;


StatementMeta(, 22a0bc23-e66a-4704-8eb7-d4f9492dfb7d, 5, Finished, Available, Finished)

<Spark SQL result set with 1 rows and 2 fields>

**2. Date range + zone coverage**

In [11]:
%%sql
SELECT
  MIN(pickup_date), MAX(pickup_date),
  MIN(pickup_hour), MAX(pickup_hour),
  COUNT(DISTINCT zone_id) AS zone_cnt
FROM gold_zone_hour_metrics;


StatementMeta(, 22a0bc23-e66a-4704-8eb7-d4f9492dfb7d, 15, Finished, Available, Finished)

<Spark SQL result set with 1 rows and 5 fields>

**3. Check distinct zones vs silver**

In [7]:
%%sql
SELECT COUNT(DISTINCT zone_id) FROM gold_zone_hour_metrics;
SELECT COUNT(DISTINCT PULocationID) FROM silver_nyc_trips_clean;

StatementMeta(, 22a0bc23-e66a-4704-8eb7-d4f9492dfb7d, 10, Finished, Available, Finished)

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 1 rows and 1 fields>

**4. Spot‑check one day**

In [16]:
%%sql
-- From day gold
SELECT
  service_date,
  zone_id,
  trips AS day_trips
FROM gold_zone_day_metrics
WHERE service_date = DATE '2025-01-15';

-- From hour gold rolled up
SELECT
  pickup_date    AS service_date,
  zone_id,
  SUM(trips_count) AS hour_trips
FROM gold_zone_hour_metrics
WHERE pickup_date = DATE '2025-01-15'
GROUP BY pickup_date, zone_id;


StatementMeta(, 22a0bc23-e66a-4704-8eb7-d4f9492dfb7d, 21, Finished, Available, Finished)

<Spark SQL result set with 222 rows and 3 fields>

<Spark SQL result set with 222 rows and 3 fields>

In [8]:
%%sql
describe silver_nyc_trips_clean

StatementMeta(, 1e2ef255-2c57-4238-a966-766c2927d980, 10, Finished, Available, Finished)

<Spark SQL result set with 22 rows and 3 fields>

In [1]:
%%sql
DESCRIBE TABLE gold_zone_hour_metrics;  
SELECT COUNT(*) AS row_count FROM gold_zone_hour_metrics;  
SELECT * FROM gold_zone_hour_metrics LIMIT 5;  


StatementMeta(, 5730f97f-3697-4386-b3f7-65b9ee86325e, 4, Finished, Available, Finished)

<Spark SQL result set with 11 rows and 3 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 5 rows and 11 fields>

In [12]:
%%sql
DESCRIBE TABLE silver_zone_hour_metrics_streaming;
select count(*) from silver_zone_hour_metrics_streaming;
select * from silver_zone_hour_metrics_streaming limit 50;

StatementMeta(, 9e67e35a-d735-499d-9466-b385f05e135c, 24, Finished, Available, Finished)

<Spark SQL result set with 19 rows and 3 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 50 rows and 19 fields>

In [15]:
%%sql

CREATE TABLE IF NOT EXISTS gold_zone_hour_streaming (
  pickup_date date,
  pickup_hour int,
  zone_id int,
  trips_count bigint,
  total_revenue double,
  avg_total_amount double,
  avg_trip_distance double,
  avg_trip_duration_min double,
  tip_amount_total double,
  tip_share_pct double,
  is_weekend boolean,
  load_timestamp timestamp
);


StatementMeta(, 9e67e35a-d735-499d-9466-b385f05e135c, 27, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

In [18]:
%%sql
INSERT INTO gold_zone_hour_streaming
WITH base AS (
  SELECT
    CAST(date_trunc('day', to_timestamp(tpep_pickup_datetime)) AS date)   AS pickup_date,
    HOUR(to_timestamp(tpep_pickup_datetime))                              AS pickup_hour,
    CAST(PULocationID AS int)                                             AS zone_id,
    trip_distance,
    total_amount,
    tip_amount,
    (UNIX_TIMESTAMP(to_timestamp(tpep_dropoff_datetime)) -
     UNIX_TIMESTAMP(to_timestamp(tpep_pickup_datetime))) / 60.0           AS trip_duration_min
  FROM silver_zone_hour_metrics_streaming
  WHERE
    PULocationID IS NOT NULL
    AND tpep_pickup_datetime IS NOT NULL
    AND tpep_dropoff_datetime IS NOT NULL
    AND total_amount > 0
)
SELECT
  pickup_date,
  pickup_hour,
  zone_id,
  COUNT(*)                                     AS trips_count,
  SUM(total_amount)                            AS total_revenue,
  AVG(total_amount)                            AS avg_total_amount,
  AVG(trip_distance)                           AS avg_trip_distance,
  AVG(trip_duration_min)                       AS avg_trip_duration_min,
  SUM(tip_amount)                              AS tip_amount_total,
  CASE WHEN SUM(total_amount) = 0 THEN 0.0
       ELSE SUM(tip_amount) / SUM(total_amount)
  END                                          AS tip_share_pct,
  CASE
    WHEN dayofweek(pickup_date) IN (1,7)       -- 1=Sunday, 7=Saturday
    THEN TRUE ELSE FALSE
  END                                          AS is_weekend,
  CURRENT_TIMESTAMP                            AS load_timestamp
FROM base
GROUP BY
  pickup_date,
  pickup_hour,
  zone_id;


StatementMeta(, 9e67e35a-d735-499d-9466-b385f05e135c, 30, Finished, Available, Finished)

<Spark SQL result set with 0 rows and 0 fields>

In [19]:
%%sql
DESCRIBE TABLE gold_zone_hour_streaming;
select count(*) from gold_zone_hour_streaming;
select * from gold_zone_hour_streaming limit 50;

StatementMeta(, 9e67e35a-d735-499d-9466-b385f05e135c, 33, Finished, Available, Finished)

<Spark SQL result set with 12 rows and 3 fields>

<Spark SQL result set with 1 rows and 1 fields>

<Spark SQL result set with 50 rows and 12 fields>